1. Show the isomorphism between Maybe a and Either () a

```haskell
maybeToEither :: Maybe a -> Either () a
maybeToEither Nothing = Left ()
maybeToEither (Just x) = Right x

eitherToMaybe :: Either () a -> Maybe a
eitherToMaybe (Left ()) = Nothing
eitherToMaybe (Right x) = Just x
```

2. Here’s a sum type defined in Haskell:
```haskell
data Shape = Circle Float | Rect Float Float
```
When we want to define a function like area that acts on a Shape,
we do it by pattern matching on the two constructors:
```haskell
area :: Shape -> Float
area (Circle r) = pi * r * r
area (Rect d h) = d * h
```
Implement Shape in C++ as an interface and create two
classes: Circle and Rect. Implement area as a virtual function.

```cpp
#include <cmath>

class Shape {
public:
    virtual float area() = 0;
};

class Circle : public Shape {
    float r;
public:
    Circle(float r) : r(r) {}
    float area() override {
        return M_PI * r * r;
    }
};

class Rect : public Shape {
    float d, h;
public:
    Rect(float d, float h) : d(d), h(h) {}
    float area() override {
        return d * h;
    }
};
```

3. Continuing with the previous example: We can easily add a new
function circ that calculates the circumference of a Shape. We
can do it without touching the definition of Shape:
```haskell
circ :: Shape -> Float
circ (Circle r) = 2.0 * pi * r
circ (Rect d h) = 2.0 * (d + h)
```
Add circ to your C++ implementation. What parts of the
original code did you have to touch?

```cpp
#include <cmath>

class Shape {
public:
    virtual float area() = 0;
};

class Circle : public Shape {
    float r;
public:
    Circle(float r) : r(r) {}
    float area() override {
        return M_PI * r * r;
    }
    float circ() override {
        return 2.0 * M_PI * r;
    }
};

class Rect : public Shape {
    float d, h;
public:
    Rect(float d, float h) : d(d), h(h) {}
    float area() override {
        return d * h;
    }
    float circ() override {
        return 2.0 * (d + h);
    }
};
```

4. Continuing further: Add a new shape, Square, to Shape and make
all the necessary updates. What code did you have to touch in
Haskell vs. C++? (Even if you’re not a Haskell programmer, the modifications should be pretty obvious.)

In [19]:
%%writefile shape.hs

data Shape = Circle Float | Rect Float Float | Square Float 
    deriving (Show)

area :: Shape -> Float
area (Circle r) = pi * r * r
area (Rect d h) = d * h
area (Square s) = s * s

circ :: Shape -> Float
circ (Circle r) = 2 * pi * r
circ (Rect d h) = 2 * (d + h)
circ (Square s) = 4 * s


main = do
    let shapes = [Circle 5.0, Rect 4.0 6.0, Square 5.0]
    mapM_ (\shape -> print "Area: %.2f\n" (area shape)) shapes

    

Overwriting shape.hs


In [20]:
!runghc shape.hs


shape.hs:18:22: error:
    • Couldn't match expected type: Float -> m b0
                  with actual type: IO ()
    • The function ‘print’ is applied to two value arguments,
        but its type ‘String -> IO ()’ has only one
      In the expression: print "Area: %.2f\n" (area shape)
      In the first argument of ‘mapM_’, namely
        ‘(\ shape -> print "Area: %.2f\n" (area shape))’
    • Relevant bindings include main :: m () (bound at shape.hs:16:1)
   |
18 |     mapM_ (\shape -> print "Area: %.2f\n" (area shape)) shapes
   |                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


Making library for the shape classes

In [1]:
%%writefile shape.h
#include <cmath>

class Shape {
public:
    virtual float area() const = 0;
    virtual float circ() const = 0;
};

class Circle : public Shape {
private:
    float radius;
public:
    Circle(float r);
    float area() const override;
    float circ() const override;
};

class Rect : public Shape {
private:
    float width, height;
public:
    Rect(float w, float h);
    float area() const override;
    float circ() const override;
};

class Square : public Shape {
private:
    float side;
public:
    Square(float s);
    float area() const override;
    float circ() const override;
};

Overwriting shape.h


In [2]:
%%writefile shape.cpp
#include "shape.h"

Circle::Circle(float r) : radius(r) {}

float Circle::area() const {
    return M_PI * radius * radius;
}

float Circle::circ() const {
    return 2.0 * M_PI * radius;
}

Rect::Rect(float w, float h) : width(w), height(h) {}

float Rect::area() const {
    return width * height;
}

float Rect::circ() const {
    return 2.0 * (width + height);
}

Square::Square(float s) : side(s) {}

float Square::area() const {
    return side * side;
}

float Square::circ() const {
    return 4.0 * side;
}

Overwriting shape.cpp


Compile into object file without linking

In [3]:
!g++ -c shape.cpp -o shape.o

In [4]:
%%writefile main.cpp
#include <iostream>
#include "shape.h"

int main() {
    Shape* shapes[] = {
        new Circle(5.0),
        new Rect(4.0, 6.0),
        new Square(5.0)
    };

    for (Shape* shape : shapes) {
        std::cout << "Area: " << shape->area() << std::endl;
        std::cout << "Circumference: " << shape->circ() << std::endl;
    }
    return 0;
}

Overwriting main.cpp


Compile and link main.cpp with the shape.o object file to create the executable

In [5]:
!g++ main.cpp shape.o -o main

In [6]:
%%script bash
./main

Area: 78.5398
Circumference: 31.4159
Area: 24
Circumference: 20
Area: 25
Circumference: 20


5. Show that ```a + a = 2 * a``` holds for types (up to isomorphism).
Remember that 2 corresponds to Bool, according to our translation table.